# Caretta Lab AMR Testing Notebook

This python notebook provides an interactive interface with the Anisotropic Magnetoresistance (AMR) testing functionality available in the open source Python Integrated Experiment Control (piec) package. The AMR measurement uses a lock-in amplifier, DC calibrator, DMM, and stepper motor to measure the angular dependence of resistance in magnetic thin films. Code documentation and setup details can be found at https://piec.readthedocs.io.

*COPY THIS NOTEBOOK AND RUN IN A LOCAL DIRECTORY TO PREVENT IT FROM BEING OVERWRITTEN BY GITHUB*

### Imports

In [ ]:
from piec.drivers.lockin.virtual_lockin import VirtualLockin
from piec.drivers.dmm.virtual_dmm import VirtualDMM
from piec.drivers.dc_calibrator.virtual_calibrator import VirtualCalibrator
from piec.drivers.stepper_motor.virtual_stepper import VirtualStepper
from piec.measurement.amr import AMR
from piec.analysis.utilities import standard_csv_to_metadata_and_data
from piec.drivers.utilities import PiecManager

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

In [ ]:
pm = PiecManager()
pm.list_resources()

### Instrument Setup

Every AMR experiment requires four instruments:
- **Lock-in Amplifier** (lockin): Drives the AC excitation and measures the in-phase (X) and quadrature (Y) voltage response.
- **DC Calibrator** (calibrator): Sets the magnetic field by outputting an analog voltage to the electromagnet power supply.
- **Digital Multimeter** (dmm): Reads back the actual field via the gaussmeter voltage output for verification.
- **Stepper Motor** (arduino): Rotates the sample platform to sweep the angle between current and magnetization.

In piec, each instrument is represented by a class object. Initialize them with the appropriate VISA address. Use `'VIRTUAL'` addresses to run in simulation mode without real hardware.

In [ ]:
lockin = VirtualLockin('VIRTUAL')     # Lock-in amplifier: 'GPIB0::8::INSTR' for real, 'VIRTUAL' for simulation
dmm = VirtualDMM('VIRTUAL')           # DMM: 'GPIB0::22::INSTR' for real, 'VIRTUAL' for simulation
calibrator = VirtualCalibrator('VIRTUAL')  # DC Calibrator: 'GPIB0::5::INSTR' for real, 'VIRTUAL' for simulation
arduino = VirtualStepper('VIRTUAL')   # Stepper motor: 'COM3' for real, 'VIRTUAL' for simulation

print('Lockin: ', lockin.idn())
print('DMM: ', dmm.idn())
print('Calibrator: ', calibrator.idn())
print('Stepper: ', arduino.idn())

### AMR Measurement

The AMR measurement rotates the sample through a range of angles while measuring the resistance via the lock-in amplifier. The AMR effect produces a cos²(θ) dependence of resistance on the angle between the current direction and the magnetization.

Key parameters:
- **field**: The applied magnetic field in Oersted (Oe), set via the DC calibrator.
- **angle_step**: The angular step size in degrees between measurements.
- **total_angle**: The total angular range to sweep (e.g., 360° for a full rotation).
- **amplitude**: The AC excitation voltage amplitude from the lock-in oscillator.
- **frequency**: The AC excitation frequency in Hz.
- **measure_time**: Time to average each data point in seconds.
- **sensitivity**: Lock-in sensitivity setting.

In [ ]:
### VARIABLE DEFINITIONS ###

path = r"scratch"  # path to save data, make sure directory exists and you have permissions

field = 1000            # applied magnetic field (Oe)
angle_step = 15         # angular step size (degrees)
total_angle = 360       # total rotation (degrees)
amplitude = 1.0         # lock-in AC excitation voltage (V)
frequency = 10          # lock-in excitation frequency (Hz)
measure_time = 1        # averaging time per data point (s) — use ~60s for real measurements
sensitivity = '50uv/pa' # lock-in sensitivity setting

In [ ]:
### RUN ###

experiment = AMR(
    dmm=dmm,
    calibrator=calibrator,
    arduino=arduino,
    lockin=lockin,
    field=field,
    angle_step=angle_step,
    total_angle=total_angle,
    amplitude=amplitude,
    frequency=frequency,
    measure_time=measure_time,
    sensitivity=sensitivity,
    save_dir=path,
    live_plot=True
)

# Run the experiment — this will set the field, configure the lock-in,
# rotate through angles capturing data, save results, and plot.
experiment.run_experiment()

In [ ]:
# Retrieve the captured data for further analysis
amr_metadata, amr_df = standard_csv_to_metadata_and_data(experiment.filename)
amr_df.head()

In [ ]:
# Plot the AMR curve: X voltage vs angle
amr_df.plot(x="angle", y=["X", "Y"], subplots=True, figsize=(10, 6), title="AMR Measurement")
plt.tight_layout()
plt.show()